# LP-FT RAD-DINO — MULTI-VIEW 7:3 (study-level) — 13 nhãn CheXpert

**Khác bản gốc:** FT fine-tune backbone **ở mức STUDY** — mỗi study forward CẢ frontal+lateral qua backbone (chia sẻ weight) → gộp `0.7·F + 0.3·L` ở mức feature (chuẩn hóa theo view có mặt) → head → ASL.
- **Tăng LR** (backbone 3e-5, head 3e-4; LP head 2e-3), **tăng epoch** (LP 40, FT 20), **TẮT early-stopping** (chạy hết epoch, vẫn lưu best + last).
- Cache mở rộng thêm **lateral** (tái dùng cache frontal đã có, chỉ tải thêm lateral).
- 13 nhãn (bỏ Support Devices), U-Ones, split 70/10/20 theo subject. Output file MỚI (không đè bản gốc).
> **Cảnh báo runtime:** FT forward 2 view/study → chậm hơn ~1.5–2× bản frontal-only; 20 epoch + tắt early-stop nên sẽ lâu. Tùy chỉnh `EPOCHS_FT`/`STUDY_BATCH` nếu cần.

## 1. Env + config

In [ ]:
import os, io, time, glob, math, shutil, random, numpy as np, pandas as pd, torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as T
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
try: drive.mount('/content/drive')
except Exception as e: print('drive:', e)
# Cần GCS: !gcloud auth application-default login --no-launch-browser  (nếu chưa auth phiên này)

# ---- ĐƯỜNG DẪN ----
SRC_DIR   = '/content/drive/MyDrive/MIMIC_CXR'          # ĐỌC: manifest_cohort.parquet + embeddings_raddino_*
WRITE_DIR = '/content/drive/MyDrive/lpft_out'           # GHI (Drive riêng)
CACHE_DIR = '/content/drive/MyDrive/lpft_cache256'      # cache uint8 256 (TÁI DÙNG frontal đã có + thêm lateral)
LOCAL_CACHE = '/content/lpft_cache256'
for d in (WRITE_DIR, CACHE_DIR, LOCAL_CACHE): os.makedirs(d, exist_ok=True)

BILLING_PROJECT='mimic-ehr-v310'; REPO='microsoft/rad-dino'; BUCKET='mimic-cxr-jpg-2.1.0.physionet.org'
THREADS=32; CACHE_RES=256; TRAIN_RES=224; CACHE_SHARD=4000; N_UNFREEZE=3
WF, WL = 0.7, 0.3                                        # trọng số frontal : lateral
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

proc=AutoImageProcessor.from_pretrained(REPO); MEAN, STD = proc.image_mean, proc.image_std
CHEXPERT_LABELS=['Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax','Support Devices']
LABELS=[c for c in CHEXPERT_LABELS if c!='Support Devices']   # 13
print('LABELS:', len(LABELS), '| WF:WL =', WF, WL)

def resolve_uones(L): return np.where(np.isnan(L),0.0,np.where(L==-1,1.0,L)).astype('float32')
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, fbeta_score
GRID=np.arange(0.05,0.96,0.01)
def bthr(y,p): s=[fbeta_score(y,(p>=t).astype(int),beta=2,zero_division=0) for t in GRID]; return float(GRID[int(np.argmax(s))])
def macro_ap_from_P(P, Lraw):
    aps=[]
    for j in range(len(LABELS)):
        col=Lraw[:,j]; keep=col!=-1; yt=np.where(np.isnan(col[keep]),0.,col[keep]).astype(int)
        if yt.sum()>0: aps.append(average_precision_score(yt, P[keep,j]))
    return float(np.mean(aps))

## 2. cm + split 70/10/20 theo subject (dùng frontal+lateral)

In [ ]:
cm = pd.read_parquet(os.path.join(SRC_DIR,'manifest_cohort.parquet'))
cm['rel'] = cm['jpg_path'].str.split(BUCKET+'/').str[-1]
print('cm:', cm.shape, '| views:', cm.view.value_counts().to_dict())

rng=np.random.default_rng(42); uniq=np.unique(cm.subject_id.values); rng.shuffle(uniq)
n=len(uniq); s1,s2=int(.70*n),int(.80*n)
tr_s,va_s,te_s=set(uniq[:s1]),set(uniq[s1:s2]),set(uniq[s2:])
def subj_split(s): return 'train' if s in tr_s else ('validate' if s in va_s else 'test')
cm['mysplit']=cm.subject_id.map(subj_split)
assert not (tr_s&te_s) and not (tr_s&va_s) and not (va_s&te_s)
print('subject 70/10/20 (ảnh):', cm.mysplit.value_counts().to_dict())

## 3. Cache 256px — frontal + lateral (resumable, tái dùng cache frontal cũ)

In [ ]:
sel = cm[cm.view.isin(['frontal','lateral'])].reset_index(drop=True)
print('cần cache (frontal+lateral):', len(sel), '|', sel.view.value_counts().to_dict())

from google.cloud import storage
import google.auth
from google.auth.transport.requests import AuthorizedSession
from requests.adapters import HTTPAdapter
from concurrent.futures import ThreadPoolExecutor
tf256=T.Compose([T.Resize(CACHE_RES), T.CenterCrop(CACHE_RES)])
_bucket=None
def _get_bucket():
    global _bucket
    if _bucket is None:
        creds,_=google.auth.default(); sess=AuthorizedSession(creds)
        sess.mount('https://', HTTPAdapter(pool_connections=THREADS,pool_maxsize=THREADS))
        _bucket=storage.Client(project=BILLING_PROJECT,credentials=creds,_http=sess).bucket(BUCKET,user_project=BILLING_PROJECT)
    return _bucket
def _one256(rel):
    try:
        im=Image.open(io.BytesIO(_get_bucket().blob(rel).download_as_bytes())).convert('RGB')
        return np.asarray(tf256(im), dtype=np.uint8)
    except Exception: return None
class BatchDS(Dataset):
    def __init__(s,df,bs): s.rel=df.rel.tolist(); s.id=df.dicom_id.tolist(); s.bs=bs
    def __len__(s): return (len(s.id)+s.bs-1)//s.bs
    def __getitem__(s,b):
        lo=b*s.bs; hi=min(lo+s.bs,len(s.id)); ids=s.id[lo:hi]; rels=s.rel[lo:hi]
        with ThreadPoolExecutor(THREADS) as ex: ims=list(ex.map(_one256,rels))
        keep=[(i,im) for i,im in zip(ids,ims) if im is not None]
        if not keep: return [],None
        return [k[0] for k in keep], np.stack([k[1] for k in keep])

done=set()
for f in glob.glob(os.path.join(CACHE_DIR,'cacheidx_*.parquet')): done|=set(pd.read_parquet(f).dicom_id)
todo=sel[~sel.dicom_id.isin(done)].reset_index(drop=True)
print('đã cache:', len(done), '| còn (chủ yếu lateral mới):', len(todo))

sh=len(glob.glob(os.path.join(CACHE_DIR,'cache_*.npy'))); buf_id=[]; buf_im=[]
def flush():
    global sh,buf_id,buf_im
    np.save(os.path.join(CACHE_DIR,f'cache_{sh:05d}.npy'), np.concatenate(buf_im,0))
    pd.DataFrame([{'dicom_id':d,'shard':sh,'row':k} for k,d in enumerate(buf_id)]
                 ).to_parquet(os.path.join(CACHE_DIR,f'cacheidx_{sh:05d}.parquet'), index=False)
    sh+=1; buf_id=[]; buf_im=[]
if len(todo):
    loader=DataLoader(BatchDS(todo,256), batch_size=None, num_workers=8, prefetch_factor=4)
    seen=0; t0=time.time(); pbar=tqdm(loader,total=len(loader))
    for ids,ims in pbar:
        if ims is None: continue
        buf_id+=list(ids); buf_im.append(ims); seen+=len(ids)
        pbar.set_postfix(img_s=round(seen/(time.time()-t0),1))
        if sum(a.shape[0] for a in buf_im)>=CACHE_SHARD: flush()
    if buf_id: flush()

cidx=pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(CACHE_DIR,'cacheidx_*.parquet')))], ignore_index=True)
cidx.to_parquet(os.path.join(CACHE_DIR,'cache_index.parquet'), index=False)
for f in sorted(glob.glob(os.path.join(CACHE_DIR,'cache_*.npy'))):
    dst=os.path.join(LOCAL_CACHE, os.path.basename(f))
    if not os.path.exists(dst): shutil.copy2(f,dst)
shutil.copy2(os.path.join(CACHE_DIR,'cache_index.parquet'), os.path.join(LOCAL_CACHE,'cache_index.parquet'))
print('cache xong:', len(cidx), 'ảnh |', len(glob.glob(os.path.join(LOCAL_CACHE,'cache_*.npy'))), 'shard local')

In [ ]:
# ---- nạp cache local (memmap) + dựng study-level ----
cidx=pd.read_parquet(os.path.join(LOCAL_CACHE,'cache_index.parquet'))
shard_arrs={int(s): np.load(os.path.join(LOCAL_CACHE,f'cache_{int(s):05d}.npy'), mmap_mode='r') for s in cidx.shard.unique()}
id2loc={d:(int(s),int(r)) for d,s,r in zip(cidx.dicom_id,cidx.shard,cidx.row)}
print('cache index:', len(id2loc), 'ảnh')

# ảnh có trong cache theo view
inc = cm[cm.view.isin(['frontal','lateral']) & cm.dicom_id.isin(id2loc)]
study2f = inc[inc.view=='frontal'].groupby('study_id').dicom_id.apply(list).to_dict()
study2l = inc[inc.view=='lateral'].groupby('study_id').dicom_id.apply(list).to_dict()

# bảng study master: study_id, subject_id, split, nhãn thô (study-level)
st = cm.drop_duplicates('study_id')[['study_id','subject_id','mysplit']+LABELS].copy()
st = st[st.study_id.isin(set(study2f)|set(study2l))].reset_index(drop=True)   # có ≥1 view trong cache
st['has_f']=st.study_id.isin(study2f).astype(float); st['has_l']=st.study_id.isin(study2l).astype(float)
print('studies:', len(st), '| có frontal:', int(st.has_f.sum()), '| có lateral:', int(st.has_l.sum()),
      '| có cả 2:', int((st.has_f.astype(bool)&st.has_l.astype(bool)).sum()))
print('split (study):', st.mysplit.value_counts().to_dict())

## 4. Pha 1 — LP (head trên frozen CLS đã gộp 7:3 ở mức study, KHÔNG scale)

In [ ]:
class ASL(nn.Module):
    def __init__(s,gn=4,gp=1,clip=.05,eps=1e-8):
        super().__init__(); s.gn,s.gp,s.clip,s.eps=gn,gp,clip,eps
    def forward(s,logits,y):
        xs=torch.sigmoid(logits); xp=xs; xn=1-xs
        if s.clip>0: xn=(xn+s.clip).clamp(max=1)
        l=y*torch.log(xp.clamp(min=s.eps))+(1-y)*torch.log(xn.clamp(min=s.eps))
        pt=xp*y+xn*(1-y); g=s.gp*y+s.gn*(1-y)
        return -(l*(1-pt)**g).mean()

# frozen CLS (mọi view) -> gộp 7:3 ở mức study
emb=np.load(os.path.join(SRC_DIR,'embeddings_raddino_ft_cls.npy'))
eidx=pd.read_parquet(os.path.join(SRC_DIR,'embeddings_raddino_index.parquet'))
erow={d:i for i,d in enumerate(eidx.dicom_id)}
inc2 = inc.copy(); inc2['erow']=inc2.dicom_id.map(erow); inc2=inc2[inc2.erow.notna()]
def study_mean(part):
    D=pd.DataFrame(emb[part.erow.astype(int).to_numpy()], index=part.study_id.values)
    return D.groupby(level=0).mean()
Ff=study_mean(inc2[inc2.view=='frontal']).reindex(st.study_id)     # (Nst,768) NaN nếu thiếu
Ll=study_mean(inc2[inc2.view=='lateral']).reindex(st.study_id)
hf=st.has_f.to_numpy()[:,None]; hl=st.has_l.to_numpy()[:,None]
Ffv=np.nan_to_num(Ff.to_numpy('float32')); Llv=np.nan_to_num(Ll.to_numpy('float32'))
E_st=((WF*Ffv*hf + WL*Llv*hl)/(WF*hf + WL*hl + 1e-8)).astype('float32')   # (Nst,768) gộp 7:3
Lraw_st=st[LABELS].to_numpy('float32'); Y_st=resolve_uones(Lraw_st)
print('E_st:', E_st.shape)

trm=(st.mysplit=='train').values; vam=(st.mysplit=='validate').values; tem=(st.mysplit=='test').values
Et=torch.tensor(E_st); Yt=torch.tensor(Y_st)
tr_i=np.where(trm)[0]; va_i=np.where(vam)[0]

head=nn.Linear(768,len(LABELS)).to(device)
opt=torch.optim.AdamW(head.parameters(), lr=2e-3, weight_decay=0.05); crit=ASL()   # LR TĂNG 1e-3->2e-3
EPOCHS_LP=40                                   # TĂNG epoch, TẮT early-stop
best=-1; best_state=None
for ep in range(EPOCHS_LP):
    head.train(); perm=np.random.default_rng(ep).permutation(tr_i)
    for i in range(0,len(perm),1024):
        b=perm[i:i+1024]; x=Et[b].to(device); y=Yt[b].to(device)
        opt.zero_grad(); loss=crit(head(x),y); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad(): Pv=torch.sigmoid(head(Et[va_i].to(device))).cpu().numpy()
    ap=macro_ap_from_P(Pv, Lraw_st[va_i])
    if ap>best: best=ap; best_state={k:v.detach().cpu().clone() for k,v in head.state_dict().items()}
    if ep%5==0 or ep==EPOCHS_LP-1: print(f'LP ep{ep:02d} val macro-AUPRC={ap:.4f} (best={best:.4f})')
head.load_state_dict(best_state); torch.save(head.state_dict(), os.path.join(WRITE_DIR,'lpft_head_lp_mv73.pt'))
print('LP best val macro-AUPRC:', round(best,4))

## 5. Pha 2 — FT study-level (backbone forward 2 view → gộp 7:3 → head), ASL, KHÔNG early-stop

In [ ]:
model=AutoModel.from_pretrained(REPO).to(device)
assert hasattr(model,'encoder') and hasattr(model.encoder,'layer') and hasattr(model,'layernorm')
for p in model.parameters(): p.requires_grad=False
for blk in model.encoder.layer[-N_UNFREEZE:]:
    for p in blk.parameters(): p.requires_grad=True
for p in model.layernorm.parameters(): p.requires_grad=True
head=nn.Linear(768,len(LABELS)).to(device); head.load_state_dict(torch.load(os.path.join(WRITE_DIR,'lpft_head_lp_mv73.pt')))
print('trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad)+sum(p.numel() for p in head.parameters()))

NORM=T.Normalize(MEAN,STD)
aug_tr=T.Compose([T.RandomResizedCrop(TRAIN_RES,scale=(0.8,1.0),ratio=(0.9,1.1)), T.ToTensor(), NORM])  # KHÔNG hflip
aug_ev=T.Compose([T.Resize(TRAIN_RES), T.CenterCrop(TRAIN_RES), T.ToTensor(), NORM])
ZERO=torch.zeros(3,TRAIN_RES,TRAIN_RES)
def load_img(did, tf): sd,rw=id2loc[did]; return tf(Image.fromarray(np.array(shard_arrs[sd][rw])))
class StudyDS(Dataset):
    def __init__(s,df,train):
        s.sid=df.study_id.tolist(); s.hf=df.has_f.to_numpy('float32'); s.hl=df.has_l.to_numpy('float32')
        s.y=resolve_uones(df[LABELS].to_numpy('float32')); s.tf=aug_tr if train else aug_ev; s.train=train
    def __len__(s): return len(s.sid)
    def __getitem__(s,i):
        sid=s.sid[i]; fids=study2f.get(sid,[]); lids=study2l.get(sid,[])
        if s.train: fi=random.choice(fids) if fids else None; li=random.choice(lids) if lids else None
        else:       fi=fids[0] if fids else None;             li=lids[0] if lids else None
        xf=load_img(fi,s.tf) if fi else ZERO.clone(); xl=load_img(li,s.tf) if li else ZERO.clone()
        return xf, xl, torch.tensor(s.hf[i]), torch.tensor(s.hl[i]), torch.tensor(s.y[i])

tr_df=st[trm].reset_index(drop=True); va_df=st[vam].reset_index(drop=True); te_df=st[tem].reset_index(drop=True)
STUDY_BATCH=32                                  # ~64 image-forward/step (≈ batch gốc 64)
trl=DataLoader(StudyDS(tr_df,True), batch_size=STUDY_BATCH, shuffle=True,  num_workers=8, pin_memory=True, drop_last=True, persistent_workers=True)
vll=DataLoader(StudyDS(va_df,False),batch_size=STUDY_BATCH*2,shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)

# LR TĂNG: backbone 1e-5->3e-5, head 1e-4->3e-4
opt=torch.optim.AdamW([{'params':[p for p in model.parameters() if p.requires_grad],'lr':3e-5},
                       {'params':head.parameters(),'lr':3e-4}], weight_decay=0.05)
EPOCHS_FT=20                                     # TĂNG epoch, TẮT early-stop
sched=torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-5,3e-4], total_steps=len(trl)*EPOCHS_FT, pct_start=0.1)
crit=ASL()
def cls(x): return model(pixel_values=x, interpolate_pos_encoding=True).last_hidden_state[:,0]
def fwd(xf,xl,hf,hl):
    F=cls(xf); L=cls(xl); hf=hf[:,None]; hl=hl[:,None]
    feat=(WF*F*hf + WL*L*hl)/(WF*hf + WL*hl + 1e-8)      # gộp 7:3, chuẩn hóa theo view có mặt
    return head(feat)
def eval_P(loader, df):
    model.eval(); head.eval(); ps=[]
    with torch.no_grad(), torch.autocast('cuda',dtype=torch.bfloat16):
        for xf,xl,hf,hl,y in loader:
            ps.append(torch.sigmoid(fwd(xf.to(device),xl.to(device),hf.to(device),hl.to(device))).float().cpu().numpy())
    P=np.concatenate(ps,0); return P, macro_ap_from_P(P, df[LABELS].to_numpy('float32'))

best=-1; best_ep=-1
for ep in range(EPOCHS_FT):
    model.train(); head.train(); t0=time.time()
    for xf,xl,hf,hl,y in tqdm(trl, desc=f'FT ep{ep}'):
        xf=xf.to(device,non_blocking=True); xl=xl.to(device,non_blocking=True)
        hf=hf.to(device); hl=hl.to(device); y=y.to(device,non_blocking=True)
        opt.zero_grad()
        with torch.autocast('cuda',dtype=torch.bfloat16): loss=crit(fwd(xf,xl,hf,hl),y)
        loss.backward(); opt.step(); sched.step()
    _,ap=eval_P(vll, va_df)
    print(f'FT ep{ep} val macro-AUPRC={ap:.4f} ({time.time()-t0:.0f}s)')
    ckpt={'backbone':model.state_dict(),'head':head.state_dict(),'val_auprc':ap,'epoch':ep,
          'labels':LABELS,'N_UNFREEZE':N_UNFREEZE,'WF':WF,'WL':WL}
    torch.save(ckpt, os.path.join(WRITE_DIR,'raddino_lpft_mv73_last.pt'))    # last (mỗi epoch)
    if ap>best:
        best=ap; best_ep=ep
        torch.save(ckpt, os.path.join(WRITE_DIR,'raddino_lpft_mv73_best.pt'))
        print('  ✓ saved BEST checkpoint')
    # KHÔNG early-stop — chạy hết EPOCHS_FT
print(f'FT xong. best val macro-AUPRC={best:.4f} @ ep{best_ep} (đã lưu best + last)')

## 6. Eval test study-level — per-label AUROC/AUPRC/precision/recall/F2

In [ ]:
ck=torch.load(os.path.join(WRITE_DIR,'raddino_lpft_mv73_best.pt'))
model.load_state_dict(ck['backbone']); head.load_state_dict(ck['head'])
tel=DataLoader(StudyDS(te_df,False), batch_size=STUDY_BATCH*2, num_workers=8, pin_memory=True)
Pte,_=eval_P(tel, te_df); Pva,_=eval_P(vll, va_df)
Lr_te=te_df[LABELS].to_numpy('float32'); Lr_va=va_df[LABELS].to_numpy('float32')
rows=[]
for j,c in enumerate(LABELS):
    cv=Lr_va[:,j]; kv=cv!=-1; yv=np.where(np.isnan(cv[kv]),0.,cv[kv]).astype(int); pv=Pva[kv,j]
    ct=Lr_te[:,j]; kt=ct!=-1; yt=np.where(np.isnan(ct[kt]),0.,ct[kt]).astype(int); pt=Pte[kt,j]
    thr=bthr(yv,pv); pred=(pt>=thr).astype(int)
    rows.append(dict(label=c,
        auroc=roc_auc_score(yt,pt) if 0<yt.sum()<len(yt) else np.nan,
        auprc=average_precision_score(yt,pt) if yt.sum()>0 else np.nan,
        precision=precision_score(yt,pred,zero_division=0),
        recall=recall_score(yt,pred,zero_division=0),
        f2=fbeta_score(yt,pred,beta=2,zero_division=0), thr=round(thr,3),
        prevalence=round(float(yt.mean()),4), n_pos=int(yt.sum())))
mt=pd.DataFrame(rows); mt.to_csv(os.path.join(WRITE_DIR,'lpft_metrics_mv73.csv'), index=False)
print(mt.to_string(index=False))
print('\nMACRO LP-FT MV7:3 (study-level):', mt[['auroc','auprc','precision','recall','f2']].mean().round(4).to_dict())
print('(so bản frontal-only image-level: test AUROC~0.797 AUPRC~0.360)')